# Objective

Illustrate fine-tuning Mistral to follow a specific output format (for e.g., JSON).

# Setup

In [ ]:
import plotly.io as pio
pio.renderers.default = "png"

In [ ]:
# #@title Run this cell to setup Unsloth on Colab
# !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install -q --no-deps xformers==0.0.28.post1 trl==0.11.1 peft==0.13.0 accelerate==0.34.2 bitsandbytes==0.44.1
# !pip install triton==3.0.0

In [ ]:
!pip install "trl<0.15.0"

Run this cell to setup Unsloth on Colab

In [ ]:
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
#download datasets evaluate rouge_score and bert score
!pip install -q datasets==3.0.1 evaluate==0.4.3 bert_score

In [ ]:
import torch
import json

from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

# Model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True
)

In [ ]:
tokenizer.add_bos_token, tokenizer.add_eos_token

In [ ]:
EOS_TOKEN = tokenizer.eos_token

# Data

In [ ]:
dataset = load_dataset("pgurazada1/entities-laptop")

In [ ]:
training_dataset = dataset['train']
validation_dataset = dataset['validation']

In [ ]:
training_dataset[0]

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [ ]:
def prompt_formatter(example, prompt_template):
    instruction="Extract entities in the input review in a JSON format."
    review=example["review"]
    entities=example["entities"]

    formatted_prompt = prompt_template.format(instruction, review, entities) + EOS_TOKEN

    return {'formatted_prompt': formatted_prompt}

In [ ]:
formatted_training_dataset = training_dataset.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

In [ ]:
formatted_validation_dataset = validation_dataset.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

In [ ]:
formatted_training_dataset[0]

# Fine-tuning

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=4,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing=True,
    random_state=42,
    loftq_config=None
)

Notice how $r = \alpha/4$.

In [ ]:
def formatting_func(example):
    # example["formatted_prompt"] can be:
    #  - a single string
    #  - a list of strings (in batched mode)

    text = example["formatted_prompt"]

    # if it is already a list -> return as-is
    if isinstance(text, list):
        return text

    # if it is a single string -> wrap inside a list
    return [text]

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_training_dataset,
    eval_dataset=formatted_validation_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    formatting_func=formatting_func,
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=20,
        eval_strategy="epoch",
        save_strategy='epoch',
        metric_for_best_model="eval_loss",
        load_best_model_at_end=True,
        greater_is_better=False,
        learning_rate=5e-5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
        output_dir="outputs"
    )
)

In [ ]:
training_history = trainer.train()

# Inference

In [ ]:
FastLanguageModel.for_inference(model)

In [ ]:
model

In [ ]:
test_review = """
This laptop impresses with its remarkable battery life, lasting well beyond the competition. The sleek design adds a touch of elegance, complemented by a comfortable keyboard that enhances productivity. Performance is stellar, seamlessly handling demanding tasks. Sturdiness is evident in its durable build, ensuring longevity. The trackpad is responsive and precise, contributing to an overall delightful user experience. In summary, this laptop excels across the board, making it a top choice for those seeking a reliable and high-performing device.
"""

In [ ]:
instruction = "Extract entities in the input review in a JSON format."

In [ ]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        instruction,
        test_review,
        "", # leave output blank for generation
    )
], return_tensors="pt").to("cuda")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id
)

In [ ]:
print(
    tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
        cleanup_tokenization_spaces=True
    )
)

# Evaluation

In [ ]:
examples = [
    "The Dell XPS 13 impresses with its sleek design and remarkable battery life. " \
    "The backlit keyboard offers a comfortable typing experience, while the powerful performance ensures seamless multitasking. " \
    "Sturdiness is top-notch, but the trackpad could be more responsive.",
    "Apple's MacBook Air M2 is a design masterpiece, ultra-thin and lightweight. "
    "The keyboard is a joy to type on, but its standout feature is the incredible battery life. "\
    "Performance-wise, it's a powerhouse, though the sturdiness could be enhanced.",
    "The HP Spectre x360 blends elegance with versatility. "\
    "The keyboard is tactile, and the battery life is commendable. "\
    "Performance-wise, it's a workhorse, but the sturdiness feels slightly compromised. "\
    "The trackpad, however, is responsive and intuitive.",
    "Lenovo's ThinkPad X1 Carbon exudes durability with its robust design. "\
    "The keyboard is superb for long typing sessions. "\
    "Although the battery life falls short of some competitors, the performance is stellar. "\
    "The trackpad is precise, but the design lacks a modern flair.",
    "The Asus ROG Zephyrus G14 is a powerhouse for gaming and productivity. "\
    "While the design is gaming-centric, the keyboard is comfortable, and the performance is exceptional. "\
    "Battery life is average, but the sturdiness is noteworthy. The trackpad could use some improvement."
]

In [ ]:
for example in examples:
  inputs = tokenizer(
      [
          alpaca_prompt.format(
              instruction,
              example,
              "", # leave output blank for generation
          )
      ], return_tensors="pt").to("cuda")

  outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id
  )

  output_json = tokenizer.decode(
      outputs[0][inputs.input_ids.shape[-1]:],
      skip_special_tokens=True,
      cleanup_tokenization_spaces=True
  )

  print(example)
  print(output_json.replace("'", '"'))
  print("***\n")

In [ ]:
# Export the model locally as a 5-bit GGUF
model.save_pretrained_gguf(
    "mistral_7b_q5_gguf",
    tokenizer,
    quantization_method = "q5_k_m"
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r /content/mistral_7b_q5_gguf_gguf /content/drive/MyDrive/